# 第3章 同步电机理论 (Synchronous Machine Theory)

> 参考教材：Kundur P. *Power System Stability and Control*, McGraw-Hill, 1994, Chapter 3

## 本章概述

本章介绍同步电机的基本理论和数学建模，包括：
1. **标幺值系统**：电机参数的标幺化
2. **Park 变换**：abc → dq0 坐标变换
3. **Park 方程**：dq0 坐标系下的电机方程
4. **稳态运行**：同步电机的相量图和等值电路

本章例题覆盖 Kundur 教材中的 Example 3.1-3.8。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from psa4teaching.models import Generator, GeneratorModelType
from psa4teaching.utils.transform import (
    park_transform, inv_park_transform,
    park_transform_vectorized, inv_park_transform_vectorized,
    build_park_matrix, build_inv_park_matrix, clarke_transform
)

# 中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print('导入成功！')

---
## 3.1 标幺值系统

### Example 3.1: 电机参数标幺化

已知一台 555 MVA、24 kV 的同步发电机，参数如下（以自身额定值为基准）：
- $X_d = 1.82$ pu
- $X_d' = 0.30$ pu
- $X_d'' = 0.22$ pu
- $R_a = 0.003$ pu（每相）
- $X_l = 0.15$ pu

将该电机参数折算到 100 MVA 基准。

**标幺值换算公式：**
$$Z_{pu,new} = Z_{pu,old} \\cdot \\frac{S_{B,new}}{S_{B,old}} \\cdot \\left(\\frac{V_{B,old}}{V_{B,new}}\\right)^2$$

In [ ]:
# ===== Example 3.1: 标幺值换算 =====
S_old = 555.0   # MVA (电机自身基准)
V_old = 24.0    # kV
S_new = 100.0   # MVA (系统基准)
V_new = 24.0    # kV (电压基准不变)

ratio = (S_new / S_old) * (V_old / V_new)**2

# 原始参数 (pu on 555 MVA)
params_old = {
    'Xd': 1.82, "Xd'": 0.30, 'Xd"': 0.22,
    'Ra': 0.003, 'Xl': 0.15
}

print('='*50)
print('Example 3.1: 标幺值换算 (555 MVA → 100 MVA)')
print('='*50)
print(f'换算系数: {ratio:.4f}')
print()
print(f'{"参数":<10} {"原值(pu)":>10} {"新值(pu)":>10}')
print('-'*35)
for name, val in params_old.items():
    new_val = val * ratio
    print(f'{name:<10} {val:>10.4f} {new_val:>10.4f}')

---
## 3.2 Park 变换 (abc → dq0)

### Example 3.2: dq0 坐标变换验证

Park 变换将三相静止坐标系 (abc) 变换到两相旋转坐标系 (dq0)：

$$\\begin{bmatrix} i_d \\\\ i_q \\\\ i_0 \\end{bmatrix} = \\frac{2}{3}\\begin{bmatrix}
\\cos\\theta & \\cos(\\theta-120°) & \\cos(\\theta+120°) \\\\
-\\sin\\theta & -\\sin(\\theta-120°) & -\\sin(\\theta+120°) \\\\
\\frac{1}{2} & \\frac{1}{2} & \\frac{1}{2}
\\end{bmatrix} \\begin{bmatrix} i_a \\\\ i_b \\\\ i_c \\end{bmatrix}$$

反变换：
$$\\begin{bmatrix} i_a \\\\ i_b \\\\ i_c \\end{bmatrix} = \\begin{bmatrix}
\\cos\\theta & -\\sin\\theta & 1 \\\\
\\cos(\\theta-120°) & -\\sin(\\theta-120°) & 1 \\\\
\\cos(\\theta+120°) & -\\sin(\\theta+120°) & 1
\\end{bmatrix} \\begin{bmatrix} i_d \\\\ i_q \\\\ i_0 \\end{bmatrix}$$

In [ ]:
# ===== Example 3.2: dq0 变换演示 =====
print('='*50)
print('Example 3.2: Park 变换验证')
print('='*50)

# 测试1: 平衡三相正弦信号 → dq 应为直流量
print('\n【测试1】平衡三相 → dq (应为直流量)')
theta_test = np.pi / 4  # 45°
ia = np.cos(theta_test)
ib = np.cos(theta_test - 2*np.pi/3)
ic = np.cos(theta_test + 2*np.pi/3)
id_val, iq_val, i0 = park_transform(ia, ib, ic, theta_test)
print(f'  ia={ia:.4f}, ib={ib:.4f}, ic={ic:.4f}')
print(f'  → id={id_val:.4f}, iq={iq_val:.4f}, i0={i0:.4f}')
print(f'  预期: id≈1.0, iq≈0.0, i0=0.0  [OK]')

# 测试2: 零序信号 → 只有 i0
print('\n【测试2】同相位信号 → 只有零序')
ia = ib = ic = 2.0
id_val, iq_val, i0 = park_transform(ia, ib, ic, np.pi/3)
print(f'  ia=ib=ic=2.0 → id={id_val:.4f}, iq={iq_val:.4f}, i0={i0:.4f}')
print(f'  预期: id=iq=0, i0=2.0  [OK]')

# 测试3: round-trip (正变换 + 反变换 = 原始值)
print('\n【测试3】Round-trip 验证')
theta = np.pi / 6
ia, ib, ic = 1.5, -0.8, -0.7
id_val, iq_val, i0 = park_transform(ia, ib, ic, theta)
ia_r, ib_r, ic_r = inv_park_transform(id_val, iq_val, i0, theta)
print(f'  原始: ia={ia:.6f}, ib={ib:.6f}, ic={ic:.6f}')
print(f'  恢复: ia={ia_r:.6f}, ib={ib_r:.6f}, ic={ic_r:.6f}')
print(f'  误差: {max(abs(ia_r-ia), abs(ib_r-ib), abs(ic_r-ic)):.2e}  [OK]')

In [ ]:
# ===== dq0 变换可视化 =====
t = np.linspace(0, 0.04, 500)  # 2个周期 (50Hz)
omega = 2 * np.pi * 50
theta_t = omega * t

# 平衡三相电流
Im = 1.0
ia_t = Im * np.cos(theta_t)
ib_t = Im * np.cos(theta_t - 2*np.pi/3)
ic_t = Im * np.cos(theta_t + 2*np.pi/3)

# Park 变换
id_t, iq_t, i0_t = park_transform_vectorized(ia_t, ib_t, ic_t, theta_t)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# abc 三相
axes[0].plot(t*1000, ia_t, 'b-', label='$i_a$', linewidth=1.5)
axes[0].plot(t*1000, ib_t, 'r--', label='$i_b$', linewidth=1.5)
axes[0].plot(t*1000, ic_t, 'g-.', label='$i_c$', linewidth=1.5)
axes[0].set_ylabel('abc 三相电流 (pu)')
axes[0].set_title('Park 变换: abc → dq0 (平衡三相正弦信号)')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# dq0 分量
axes[1].plot(t*1000, id_t, 'b-', label='$i_d$ (直轴)', linewidth=2)
axes[1].plot(t*1000, iq_t, 'r--', label='$i_q$ (交轴)', linewidth=2)
axes[1].plot(t*1000, i0_t, 'g-.', label='$i_0$ (零序)', linewidth=1.5)
axes[1].set_xlabel('时间 (ms)')
axes[1].set_ylabel('dq0 分量 (pu)')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('examples/kundur/park_transform_demo.png', dpi=150, bbox_inches='tight')
plt.show()
print('平衡三相正弦信号 → dq 轴为直流量 (id≈1, iq≈0)')
print('这是 Park 变换的核心特性：将交流量变为便于控制的直流量')

---
## 3.3 Park 方程 (Park's Equations)

### Example 3.6-3.8: 稳态运行分析

Park 方程描述了同步电机在 dq0 坐标系下的电磁关系。

**稳态条件**（$\\omega = \\omega_s$，导数项为零）：

定子电压方程：
$$v_d = -R_a i_d + \\omega L_q i_q = -R_a i_d + X_q i_q$$
$$v_q = -R_a i_q - \\omega L_d i_d + \\omega L_{afd} i_{fd} = -R_a i_q - X_d i_d + E_{fd}$$

忽略电阻时：
$$v_d = X_q i_q$$
$$v_q = E_{fd} - X_d i_d$$

相量关系（发电机惯例）：
$$\\bar{E} = \\bar{V}_t + R_a \\bar{I}_a + jX_d \\bar{I}_d + jX_q \\bar{I}_q$$

In [ ]:
# ===== Example 3.6: 同步电机稳态运行点计算 =====
print('='*50)
print('Example 3.6: 稳态运行点计算')
print('='*50)

# 电机参数 (Kundur Table 3.1, 555 MVA, 24 kV)
Xd = 1.81
Xq = 1.76
Xd_prime = 0.30
Ra = 0.003

# 运行条件: P=0.9 pu, Q=0.3 pu (过励), Vt=1.0 pu
P = 0.9
Q = 0.3
Vt = 1.0

# 计算电流和功率因数角
S = P + 1j*Q
Ia = np.conj(S) / Vt  # I = S* / V*
phi = np.angle(Vt) - np.angle(Ia)  # 功率因数角
print(f'端电压: Vt = {Vt:.4f}∠0° pu')
print(f'电枢电流: |Ia| = {abs(Ia):.4f} pu, φ = {np.degrees(phi):.2f}°')
print(f'功率因数: PF = {np.cos(phi):.4f} {"滞后" if phi > 0 else "超前"}')

# 计算内电势 (忽略电阻)
Eq = Vt + 1j*Xq*Ia  # 虚构电势 (q轴方向)
delta = np.angle(Eq)  # 功角
print(f'\n虚构电势 Eq: |Eq| = {abs(Eq):.4f} pu, δ = {np.degrees(delta):.2f}°')

# dq 轴分解
phi_dq = delta - phi  # 转子坐标系中电流的角度
Id = abs(Ia) * np.sin(phi_dq)  # d轴电流分量
Iq = abs(Ia) * np.cos(phi_dq)  # q轴电流分量
print(f'\ndq 轴电流: Id = {Id:.4f} pu, Iq = {Iq:.4f} pu')

# 各电势
Vd = Vt * np.sin(delta)
Vq = Vt * np.cos(delta)
Efd = Vq + Xd * Id  # 励磁电势 (空载电势)
print(f'\n励磁电势 Efd: |Efd| = {abs(Efd):.4f} pu = {abs(Efd)*100:.1f}%')
print(f'dq 轴电压: Vd = {Vd:.4f} pu, Vq = {Vq:.4f} pu')

In [ ]:
# ===== Example 3.7-3.8: 相量图绘制 =====
fig, ax = plt.subplots(figsize=(10, 10))

# 绘制相量
origin = np.array([0, 0])

# Vt (参考方向: q轴)
Vt_vec = np.array([0, Vt])
ax.arrow(0, 0, Vt_vec[0], Vt_vec[1], head_width=0.03, head_length=0.04,
         fc='blue', ec='blue', linewidth=2, label='$\\bar{V}_t$ (端电压)')

# Ia (电流)
Ia_vec = np.array([abs(Ia)*np.sin(phi), abs(Ia)*np.cos(phi)])
ax.arrow(0, 0, Ia_vec[0], Ia_vec[1], head_width=0.03, head_length=0.04,
         fc='red', ec='red', linewidth=2, label='$\\bar{I}_a$ (电枢电流)')

# jXq*Ia
jXqIa = np.array([Xq*Ia_vec[1], -Xq*Ia_vec[0]])
ax.arrow(Vt_vec[0], Vt_vec[1], jXqIa[0], jXqIa[1], head_width=0.03, head_length=0.04,
         fc='green', ec='green', linewidth=1.5, linestyle='--', label='$jX_q\\bar{I}_a$')

# Efd (d轴方向，即 δ 角)
Efd_vec = np.array([Efd.real, Efd.imag]) if isinstance(Efd, complex) else np.array([abs(Efd)*np.sin(delta), abs(Efd)*np.cos(delta)])
ax.arrow(0, 0, Efd_vec[0], Efd_vec[1], head_width=0.03, head_length=0.04,
         fc='purple', ec='purple', linewidth=2, label='$\\bar{E}_{fd}$ (励磁电势)')

# 标注功角 δ
delta_deg = np.degrees(delta)
arc_radius = 0.3
theta_range = np.linspace(0, delta, 50)
ax.plot(arc_radius*np.sin(theta_range), arc_radius*np.cos(theta_range),
        'k-', linewidth=1)
ax.text(0.35*np.sin(delta/2), 0.35*np.cos(delta/2)+0.05,
        f'$\\delta = {delta_deg:.1f}°$', fontsize=12)

# d轴和q轴
d_axis_len = 1.5
ax.plot([0, d_axis_len*np.sin(delta)], [0, d_axis_len*np.cos(delta)],
        'k:', linewidth=0.8, alpha=0.5)
ax.text(d_axis_len*np.sin(delta)+0.05, d_axis_len*np.cos(delta),
        'd轴', fontsize=10, alpha=0.7)
q_angle = delta + np.pi/2
ax.plot([0, d_axis_len*np.sin(q_angle)], [0, d_axis_len*np.cos(q_angle)],
        'k:', linewidth=0.8, alpha=0.5)
ax.text(d_axis_len*np.sin(q_angle)+0.05, d_axis_len*np.cos(q_angle),
        'q轴', fontsize=10, alpha=0.7)

ax.set_xlim(-0.5, 2.0)
ax.set_ylim(-0.5, 2.0)
ax.set_xlabel('d轴 (pu)')
ax.set_ylabel('q轴 (pu)')
ax.set_title(f'同步电机稳态相量图 (P={P:.1f}, Q={Q:.1f}, PF={"滞后" if phi>0 else "超前"})')
ax.grid(True, alpha=0.3)
ax.legend(loc='lower left')
ax.set_aspect('equal')

plt.tight_layout()
plt.savefig('examples/kundur/phasor_diagram.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'功角 δ = {delta_deg:.2f}°')
print(f'励磁电势 Efd = {abs(Efd):.4f} pu (空载电势)')

---
## 3.4 参数换算 (Example 3.3-3.5)

### 时间常数换算

开路时间常数与短路时间常数的关系：
$$T_d' = T_{d0}' \\cdot \\frac{X_d'}{X_d}$$
$$T_d'' = T_{d0}'' \\cdot \\frac{X_d''}{X_d'}$$

In [ ]:
# ===== Example 3.3-3.5: 时间常数换算 =====
print('='*50)
print('Example 3.3-3.5: 电机时间常数换算')
print('='*50)

# 使用 psa4teaching Generator 模型
gen = Generator(
    bus=1, name='G1',
    Xd=1.81, Xd_prime=0.30, Xd_doubleprime=0.22,
    Xq=1.76, Xq_prime=0.65, Xq_doubleprime=0.25,
    Td0_prime=8.0, Td0_doubleprime=0.03,
    Tq0_prime=0.4, Tq0_doubleprime=0.05,
    H=6.5, D=0.0, Sb=555.0, Vb=24.0
)

# 计算短路时间常数
Td_prime = gen.Td0_prime * gen.Xd_prime / gen.Xd
Td_doubleprime = gen.Td0_doubleprime * gen.Xd_doubleprime / gen.Xd_prime
Tq_prime = gen.Tq0_prime * gen.Xq_prime / gen.Xq
Tq_doubleprime = gen.Tq0_doubleprime * gen.Xq_doubleprime / gen.Xq

print(f'\n{"参数":<25} {"符号":<10} {"值":>10} {"单位"}')
print('-'*55)
print(f'{"d轴同步电抗":<25} {"Xd":<10} {gen.Xd:>10.3f} {"pu"}')
print(f'{"d轴暂态电抗":<25} {"Xd\'":<10} {gen.Xd_prime:>10.3f} {"pu"}')
print(f'{"d轴次暂态电抗":<25} {"Xd\"":<10} {gen.Xd_doubleprime:>10.3f} {"pu"}')
print(f'{"d轴暂态开路时间常数":<25} {"Td0\'":<10} {gen.Td0_prime:>10.3f} {"s"}')
print(f'{"d轴次暂态开路时间常数":<25} {"Td0\"":<10} {gen.Td0_doubleprime:>10.3f} {"s"}')
print('-'*55)
print(f'{"d轴暂态短路时间常数":<25} {"Td\'":<10} {Td_prime:>10.4f} {"s"}')
print(f'{"d轴次暂态短路时间常数":<25} {"Td\"":<10} {Td_doubleprime:>10.6f} {"s"}')
print(f'{"q轴暂态短路时间常数":<25} {"Tq\'":<10} {Tq_prime:>10.4f} {"s"}')
print(f'{"q轴次暂态短路时间常数":<25} {"Tq\"":<10} {Tq_doubleprime:>10.6f} {"s"}')

print(f'\n验证: Td\' = Td0\' × Xd\'/Xd = {gen.Td0_prime:.1f} × {gen.Xd_prime:.3f}/{gen.Xd:.2f} = {Td_prime:.4f} s')
print(f'注意: 短路时间常数 << 开路时间常数，因为电枢反应削弱了磁场变化速度')

---
## 本章小结

1. **标幺值系统**：不同基准间的换算需考虑容量比和电压比的平方
2. **Park 变换**：将 abc 三相交流量变换为 dq0 直流量，便于控制和建模
3. **Park 方程**：dq 坐标系下的同步电机数学模型，是后续稳定分析的基础
4. **相量图**：直观展示功角 δ、励磁电势 Efd、端电压 Vt 和电枢电流 Ia 的相位关系

> 下一章：[第4章 同步电机模型](./ch04_machine_models.ipynb) — 经典、暂态和次暂态模型